In [1]:
import cv2
import mediapipe as mp
from ultralytics import YOLO
from mediapipe.python.solutions import pose

In [2]:
person_model = YOLO("yolov8s.pt")
weapon_model = YOLO (r"runs\detect\weapon_model\weights\best.pt")

In [3]:
mp_pose = pose
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

In [4]:
# pip install mediapipe==0.10.9

In [5]:
video_path = "Prediction1.avi"
# video_path = r"C:\Users\user\Desktop\ML_PROJECT\Smart_AI_Surveillance_and_Behavior_Analysis_System\Prediction1.avi"
cap = cv2.VideoCapture(video_path)

In [6]:
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(
    "output_pose.mp4",
    fourcc,
    fps,
    (frame_width, frame_height)
)

In [7]:
cv2.namedWindow("AI Smart Surveillance System", cv2.WINDOW_NORMAL)
cv2.resizeWindow("AI Smart Surveillance System", 1000, 700)

In [8]:
armed_ids = set()

In [9]:
def detect_posture(landmarks):

    left_shoulder = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value]
    right_shoulder = landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value]
    left_hip = landmarks[mp_pose.PoseLandmark.LEFT_HIP.value]
    right_hip = landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value]
    left_wrist = landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value]
    right_wrist = landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value]

    # Average vertical positions
    shoulder_y = (left_shoulder.y + right_shoulder.y) / 2
    hip_y = (left_hip.y + right_hip.y) / 2

    # If shoulders and hips nearly same height → fallen
    if abs(shoulder_y - hip_y) < 0.08:
        return "FALLEN"

    # If wrists above shoulders → hands up
    if left_wrist.y < left_shoulder.y and right_wrist.y < right_shoulder.y:
        return "HANDS UP"

    return "STANDING"

In [10]:
while True:
    ret, frame = cap.read()
    print("Frame read status:", ret)
    if not ret:
        break

    person_results = person_model.track(frame, persist=True, conf=0.4,classes=[0], verbose=False)
    weapon_results = weapon_model(frame, conf=0.55, verbose=False)

    persons = []
    weapons = []

    # Collect persons with ID
    if person_results[0].boxes.id is not None:
        for box, track_id in zip(
                person_results[0].boxes.xyxy,
                person_results[0].boxes.id):

            x1, y1, x2, y2 = map(int, box)
            track_id = int(track_id)
            persons.append((track_id, x1, y1, x2, y2))

    # Collect weapons
    for box in weapon_results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        weapons.append((x1, y1, x2, y2))

    # Associate weapons
    for wx1, wy1, wx2, wy2 in weapons:
        cx = (wx1 + wx2) // 2
        cy = (wy1 + wy2) // 2

        for track_id, px1, py1, px2, py2 in persons:
            if px1 <= cx <= px2 and py1 <= cy <= py2:
                armed_ids.add(track_id)
    

    for wx1, wy1, wx2, wy2 in weapons:
        cv2.rectangle(frame, (wx1, wy1), (wx2, wy2),
                      (0, 255, 255), 2)
        cv2.putText(frame, "Weapon",
                    (wx1, wy1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 255), 2)
        

    # Draw persons + run MediaPipe
    for track_id, px1, py1, px2, py2 in persons:

        person_roi = frame[py1:py2, px1:px2]

        posture = "UNKNOWN"

        if person_roi.size > 0:
            roi_rgb = cv2.cvtColor(person_roi, cv2.COLOR_BGR2RGB)
            results = pose.process(roi_rgb)

            if results.pose_landmarks:
                posture = detect_posture(results.pose_landmarks.landmark)

        # Color logic
        if track_id in armed_ids:
            color = (0, 0, 255)   # RED
        else:
            color = (0, 255, 0)   # GREEN

        cv2.rectangle(frame, (px1, py1), (px2, py2), color, 2)

        label = f"ID {track_id} | {posture}"
        cv2.putText(frame, label, (px1, py1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        
    out.write(frame)

    cv2.imshow("AI Smart Surveillance System", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()
out.release()



Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read status: True
Frame read statu